# M5.S3 Modules, Apptainer and reproducible runs

A module selects software installed on the cluster. A container provides a packaged user-space environment. We will use both, inspect what actually runs, and check the results.

You have two routes through each step: type the displayed commands in SSH, or execute the notebook cell containing the same commands. Run All follows the practical in order. All calculations run on allocated CPU resources. No GPU, package installation or image download is needed.

The main container example trains one small PyTorch model. We run it twice with the same inputs, change one input, and then use an explicit bind mount. Keep a short record in `observations.txt`.

## 1. Create your working folder

In your computer's terminal, replace `YOUR_USERNAME` and connect:

```bash
ssh YOUR_USERNAME@ssh.iesci.tech
mkdir -p ~/hpc-labs/m5s3-basics
cd ~/hpc-labs/m5s3-basics
```

The notebook cell below selects that same folder. A `%%writefile` cell saves the displayed source. In SSH, use `nano filename` and paste the text below that first line.

Use the login shell for the displayed commands. Slurm starts the calculations on allocated nodes. If resources are busy, the output shows that the request is waiting.

In [ ]:
from pathlib import Path
import os

folder = Path.home() / 'hpc-labs' / 'm5s3-basics'
folder.mkdir(parents=True, exist_ok=True)
os.chdir(folder)

# Reset status records so an earlier run cannot appear as this run's success.
for stage in ['modules', 'native', 'restore', 'hdf5', 'container', 'repeat', 'changed', 'bind']:
    _ = Path(stage + '_status.txt').write_text('NOT RUN\n')
for name in ['run_A.json', 'run_B.json', 'run_C.json', 'run_bind.json', 'from_container.txt']:
    Path(name).unlink(missing_ok=True)
print('Working folder:', folder, flush=True)

### Run shell commands from Jupyter

This small helper sends commands to a shell and prints the output. It contains no scientific calculation. It removes the notebook's inherited Slurm job limits from the child process, so each exercise can request its own resources. Each cell starts a fresh shell; a module loaded in one cell does not automatically carry over to the next cell or change the notebook's Python interpreter.

In [ ]:
import subprocess
import selectors

def run(command, check=True, show_command=True):
    if show_command: print('$ ' + command, flush=True)
    # Jupyter can itself be inside a small Slurm job. Each exercise requests
    # its own allocation; only the child shell loses inherited job/step limits.
    keep = {'SLURM_CONF', 'SLURM_CONF_SERVER', 'SLURM_CLUSTERS',
            'SLURM_ACCOUNT', 'SLURM_QOS'}
    environment = {key: value for key, value in os.environ.items()
                   if not key.startswith('SLURM_') or key in keep}
    process = subprocess.Popen(['bash', '-lc', 'set -euo pipefail\n' + command],
                               env=environment, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    # Print Slurm/compiler output as it arrives, including while a job queues.
    reader = selectors.DefaultSelector()
    reader.register(process.stdout, selectors.EVENT_READ)
    while process.poll() is None:
        if reader.select(timeout=15):
            line = process.stdout.readline()
            if line: print(line, end='', flush=True)
        else:
            print('Still waiting for the command. Slurm resources may be queued.', flush=True)
    for line in process.stdout: print(line, end='', flush=True)
    reader.close()
    if process.wait() != 0:
        print('Command failed. Inspect the output above.', flush=True)
        if check: raise RuntimeError('The checkpoint did not pass.')

## 2. Inspect the available modules

Start by looking, before loading anything. `module list` shows the selected modules. `module avail` shows matching modules currently visible; `module spider` searches the catalogue, including software that needs another module loaded first.

Run these commands in SSH, or execute the matching cell below.

```bash
pwd
hostname
module list
module avail foss OpenMPI HDF5
module spider OpenMPI
module spider HDF5
command -v python3
python3 --version
```

In [ ]:
run(r"""
pwd
hostname
module list
module avail foss OpenMPI HDF5
module spider OpenMPI
module spider HDF5
command -v python3
python3 --version
""")

### Checkpoint 1

Record the Python path and find an advertised compiler/MPI stack and an HDF5 version. An empty module list does not mean the machine has no software: some programs may already be on the default path.

## 3. Load a stack, then compile a C program

We use `foss/2023b`, which was observed in an earlier SciTech run. It selects a coherent compiler and scientific-library stack. `module purge` clears the optional selection in this shell before loading that stack; it does not uninstall anything.

Read the short C program. It prints the compiler that built it and computes a sum with a known answer. This gives us something concrete to check after selecting software.

In [ ]:
%%writefile module_demo.c
#include <stdio.h>

int main(void) {
    /* Use a wide integer because the sum is larger than a 32-bit signed int. */
    long long total = 0;

    for (int number = 1; number <= 100000; number++) {
        total += number;
    }

    /* __VERSION__ identifies the compiler that built this executable. */
    printf("Compiler: %s\n", __VERSION__);
    printf("Sum 1..100000: %lld\n", total);
    return 0;
}

Run these commands in SSH, or execute the matching cell below.

```bash
module purge
module load foss/2023b
module list
command -v gcc
gcc --version | head -1
command -v mpicc
gcc -O2 -Wall module_demo.c -o module_demo
module -t list 2> selected-modules.txt
sha256sum module_demo.c > source.sha256
echo PASS > modules_status.txt
echo 'Compiled module_demo.c with the selected stack.'
```

In [ ]:
run(r"""
module purge
module load foss/2023b
module list
command -v gcc
gcc --version | head -1
command -v mpicc
gcc -O2 -Wall module_demo.c -o module_demo
module -t list 2> selected-modules.txt
sha256sum module_demo.c > source.sha256
echo PASS > modules_status.txt
echo 'Compiled module_demo.c with the selected stack.'
""")

### Execute on an allocated core

Compilation selected the compiler. `srun` now allocates a CPU core and starts the executable. The terminal and notebook use the same command.

Run these commands in SSH, or execute the matching cell below.

```bash
module purge
module load foss/2023b
echo FAILED > native_status.txt
srun -p cpu -N1 -n1 -c1 --mem=1G --time=00:02:00 ./module_demo | tee module_result.txt
grep -qx 'Sum 1..100000: 5000050000' module_result.txt
echo PASS > native_status.txt
```

In [ ]:
run(r"""
module purge
module load foss/2023b
echo FAILED > native_status.txt
srun -p cpu -N1 -n1 -c1 --mem=1G --time=00:02:00 ./module_demo | tee module_result.txt
grep -qx 'Sum 1..100000: 5000050000' module_result.txt
echo PASS > native_status.txt
""")

### Checkpoint 2

The sum must be `5000050000`. The compiler line comes from the executable, not from a guessed version. Why does the program use `long long` rather than a 32-bit signed integer?

## 4. Save and restore a module selection

A named module collection lets us request the same selection again. This step saves the course stack as `m5s3_class`, purges it, then restores it. We compare the compiler path before and after. Repeating this cell updates that named collection in your account.

The executable and result file stay in your folder when modules are unloaded. An application linked to extra shared libraries may still need its original module selection to run.

Run these commands in SSH, or execute the matching cell below.

```bash
module purge
module load foss/2023b
echo FAILED > restore_status.txt
command -v gcc > compiler-before.txt
module save m5s3_class
module purge
module list
module restore m5s3_class
module list
command -v gcc > compiler-after.txt
cmp compiler-before.txt compiler-after.txt
cat module_result.txt
echo PASS > restore_status.txt
```

In [ ]:
run(r"""
module purge
module load foss/2023b
echo FAILED > restore_status.txt
command -v gcc > compiler-before.txt
module save m5s3_class
module purge
module list
module restore m5s3_class
module list
command -v gcc > compiler-after.txt
cmp compiler-before.txt compiler-after.txt
cat module_result.txt
echo PASS > restore_status.txt
""")

### Checkpoint 3

The restored compiler path must match. Explain the difference between a saved module selection and a container image. What happens if a saved module version is later removed by the cluster administrator?

## 5. Inspect a scientific library module

HDF5 stores structured scientific data. Its compiler wrappers add the include/library paths needed by programs that use HDF5. This step inspects those wrappers; it does not perform an HDF5 calculation or install the library.

The named module comes from the earlier course practical. If it is unavailable, keep that observation and inspect `module spider HDF5` for the site's current choices.

Run these commands in SSH, or execute the matching cell below.

```bash
module purge
module load foss/2023b
echo SKIPPED > hdf5_status.txt
if module load HDF5/1.14.3-gompi-2023b; then
    echo FAILED > hdf5_status.txt
    module list
    if command -v h5cc || command -v h5pcc; then
        echo PASS > hdf5_status.txt
    fi
else
    echo 'HDF5 inspection skipped: the named module is unavailable.'
fi
```

In [ ]:
run(r"""
module purge
module load foss/2023b
echo SKIPPED > hdf5_status.txt
if module load HDF5/1.14.3-gompi-2023b; then
    echo FAILED > hdf5_status.txt
    module list
    if command -v h5cc || command -v h5pcc; then
        echo PASS > hdf5_status.txt
    fi
else
    echo 'HDF5 inspection skipped: the named module is unavailable.'
fi
""", check=False)

## 6. Inspect Apptainer and the existing image

We now use the shared PyTorch image already named in the course material. The `.sif` file packages the environment; it does not reserve resources. Although its filename mentions CUDA, this exercise uses CPU-only PyTorch and does not need `--nv`.

Save the two paths below. If your instructor supplies a different approved image, change these values together with its documented interpreter path. Do not download or rebuild an image to hide a failed check.

In [ ]:
%%writefile container.env
# Shared course image. Inspect this path; do not download or rebuild the image.
IMAGE=/data/software/containers/pytorch-2.8.0-cuda12.8.sif
# This is the interpreter inside that image, not the notebook interpreter.
CONTAINER_PYTHON=/opt/conda/bin/python3

Run these commands in SSH, or execute the matching cell below.

```bash
source container.env
echo SKIPPED > container_status.txt
if command -v apptainer && [ -r "$IMAGE" ]; then
    echo FAILED > container_status.txt
    apptainer --version
    ls -lh "$IMAGE"
    apptainer inspect "$IMAGE" > image-labels.txt
    srun -p cpu -N1 -n1 -c2 --mem=2G --time=00:03:00         apptainer exec --cleanenv "$IMAGE" "$CONTAINER_PYTHON" -c         'import sys,torch; print("Python:",sys.executable); print("PyTorch:",torch.__version__); print("CUDA available:",torch.cuda.is_available())'
    echo READY > container_status.txt
else
    echo 'Container steps skipped: Apptainer or the shared image is unavailable.'
    echo 'Inspect module avail Apptainer and the supplied image path.'
fi
```

In [ ]:
run(r"""
source container.env
echo SKIPPED > container_status.txt
if command -v apptainer && [ -r "$IMAGE" ]; then
    echo FAILED > container_status.txt
    apptainer --version
    ls -lh "$IMAGE"
    apptainer inspect "$IMAGE" > image-labels.txt
    srun -p cpu -N1 -n1 -c2 --mem=2G --time=00:03:00         apptainer exec --cleanenv "$IMAGE" "$CONTAINER_PYTHON" -c         'import sys,torch; print("Python:",sys.executable); print("PyTorch:",torch.__version__); print("CUDA available:",torch.cuda.is_available())'
    echo READY > container_status.txt
else
    echo 'Container steps skipped: Apptainer or the shared image is unavailable.'
    echo 'Inspect module avail Apptainer and the supplied image path.'
fi
""", check=False)

### Checkpoint 4

The printed Python path and PyTorch version must come from inside the image. The check runs on an allocated CPU node, so a readable image on the notebook host is not our only evidence. `CUDA available: False` is acceptable for this CPU exercise.

`--cleanenv` reduces inherited host environment settings. It does not make the container a virtual machine or hide every host folder. A working container also depends on the host kernel and runtime.

## 7. Train one small model in the container

The data follow approximately `y = 3x + 2`. The model learns a slope and an intercept. We fix the seed, limit CPU threads to two and use deterministic PyTorch operations.

Read the training loop: clear old gradients, predict, calculate the loss, calculate gradients, then update the parameters. The assertions check that training learned the intended line. Only the container needs PyTorch; the notebook interpreter does not.

In [ ]:
%%writefile train_model.py
"""Fit a straight line inside the existing PyTorch container."""
import json
import sys
import torch

# Read the two experiment parameters from the command line.
seed = int(sys.argv[1])
true_slope = float(sys.argv[2])

# Fix the random initialisation and keep the CPU workload within our allocation.
torch.manual_seed(seed)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)

# Generate the same 256 observations whenever the slope parameter is unchanged.
x = torch.linspace(-2, 2, 256).reshape(-1, 1)
y = true_slope * x + 2.0 + 0.05 * torch.sin(9 * x)

# The model learns one slope and one intercept.
model = torch.nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(240):
    optimizer.zero_grad()                 # Discard the previous gradients.
    prediction = model(x)
    loss = torch.mean((prediction - y) ** 2)
    loss.backward()                      # Calculate the new gradients.
    optimizer.step()                     # Update the slope and intercept.

# Inspect the learned values without creating more gradient calculations.
with torch.no_grad():
    slope = model.weight.item()
    intercept = model.bias.item()
    error = torch.mean(torch.abs(model(x) - y)).item()

# A program finishing is not enough: check that it learned the intended line.
assert abs(slope - true_slope) < 0.1
assert abs(intercept - 2.0) < 0.1
assert error < 0.1

# Stable, sorted output lets us compare separate executions with cmp.
result = {
    "torch_version": torch.__version__,
    "seed": seed,
    "target_slope": true_slope,
    "learned_slope": round(slope, 6),
    "learned_intercept": round(intercept, 6),
    "mean_absolute_error": round(error, 6),
}
print(json.dumps(result, sort_keys=True))

Run these commands in SSH, or execute the matching cell below.

```bash
source container.env
if [ "$(cat container_status.txt)" = READY ]; then
    echo FAILED > repeat_status.txt
    srun -p cpu -N1 -n1 -c2 --mem=2G --time=00:03:00         apptainer exec --cleanenv "$IMAGE" "$CONTAINER_PYTHON" - 42 3.0         < train_model.py | tee run_A.json
    srun -p cpu -N1 -n1 -c2 --mem=2G --time=00:03:00         apptainer exec --cleanenv "$IMAGE" "$CONTAINER_PYTHON" - 42 3.0         < train_model.py | tee run_B.json
    cmp run_A.json run_B.json
    sha256sum train_model.py run_A.json run_B.json > run-identities.txt
    echo PASS > repeat_status.txt
    echo 'PASS: separate executions A and B produced identical records.'
else
    echo SKIPPED > repeat_status.txt
    echo 'Training skipped: the container environment did not pass its check.'
fi
```

In [ ]:
run(r"""
source container.env
if [ "$(cat container_status.txt)" = READY ]; then
    echo FAILED > repeat_status.txt
    srun -p cpu -N1 -n1 -c2 --mem=2G --time=00:03:00         apptainer exec --cleanenv "$IMAGE" "$CONTAINER_PYTHON" - 42 3.0         < train_model.py | tee run_A.json
    srun -p cpu -N1 -n1 -c2 --mem=2G --time=00:03:00         apptainer exec --cleanenv "$IMAGE" "$CONTAINER_PYTHON" - 42 3.0         < train_model.py | tee run_B.json
    cmp run_A.json run_B.json
    sha256sum train_model.py run_A.json run_B.json > run-identities.txt
    echo PASS > repeat_status.txt
    echo 'PASS: separate executions A and B produced identical records.'
else
    echo SKIPPED > repeat_status.txt
    echo 'Training skipped: the container environment did not pass its check.'
fi
""", check=False)

### Checkpoint 5

The learned slope should be near 3, the intercept near 2, and the mean absolute error below 0.1. Runs A and B are separate container processes. `cmp` must succeed for the recorded outputs to match.

`< train_model.py` passes the code through standard input. The host shell saves the output with `tee`; no bind mount is needed for this particular route. A fixed seed alone does not guarantee identical results on all hardware or with different libraries. If outputs differ, inspect the recorded settings rather than claiming reproducibility.

## 8. Change one input

Keep the program, seed and image fixed. Change only the target slope from 3 to 4. The slope learned by the model should change, while the intercept stays near 2. The assertions in the program still check the numerical result.

Run these commands in SSH, or execute the matching cell below.

```bash
source container.env
if [ "$(cat container_status.txt)" = READY ] && [ "$(cat repeat_status.txt)" = PASS ]; then
    echo FAILED > changed_status.txt
    srun -p cpu -N1 -n1 -c2 --mem=2G --time=00:03:00         apptainer exec --cleanenv "$IMAGE" "$CONTAINER_PYTHON" - 42 4.0         < train_model.py | tee run_C.json
    if cmp -s run_A.json run_C.json; then
        echo 'FAIL: changing the target slope should change the result.'
        exit 1
    fi
    echo PASS > changed_status.txt
    echo 'PASS: the new input produced a different, checked result.'
else
    echo SKIPPED > changed_status.txt
    echo 'Changed-input experiment skipped: the checked baseline is not ready.'
fi
```

In [ ]:
run(r"""
source container.env
if [ "$(cat container_status.txt)" = READY ] && [ "$(cat repeat_status.txt)" = PASS ]; then
    echo FAILED > changed_status.txt
    srun -p cpu -N1 -n1 -c2 --mem=2G --time=00:03:00         apptainer exec --cleanenv "$IMAGE" "$CONTAINER_PYTHON" - 42 4.0         < train_model.py | tee run_C.json
    if cmp -s run_A.json run_C.json; then
        echo 'FAIL: changing the target slope should change the result.'
        exit 1
    fi
    echo PASS > changed_status.txt
    echo 'PASS: the new input produced a different, checked result.'
else
    echo SKIPPED > changed_status.txt
    echo 'Changed-input experiment skipped: the checked baseline is not ready.'
fi
""", check=False)

### Checkpoint 6

Which parts of the experiment stayed fixed? Why would a matching filename alone be weak evidence that two experiments used identical code or data?

## 9. Use a bind mount to share files

A bind mount maps a host directory to a path inside the container. Here our lab folder becomes `/work`. `--pwd /work` selects that working directory inside the process.

First run the same training file by its mounted path instead of sending it through standard input. Then write a small file from inside the container and read it from the host. That file is stored in your lab folder, not added to the SIF image.

Run these commands in SSH, or execute the matching cell below.

```bash
source container.env
if [ "$(cat container_status.txt)" = READY ] && [ "$(cat repeat_status.txt)" = PASS ]; then
    echo FAILED > bind_status.txt
    srun -p cpu -N1 -n1 -c2 --mem=2G --time=00:03:00         apptainer exec --cleanenv --bind "$PWD:/work" --pwd /work         "$IMAGE" "$CONTAINER_PYTHON" train_model.py 42 3.0 | tee run_bind.json
    cmp run_A.json run_bind.json
    srun -p cpu -N1 -n1 -c1 --mem=1G --time=00:02:00         apptainer exec --cleanenv --bind "$PWD:/work" --pwd /work         "$IMAGE" "$CONTAINER_PYTHON" - <<'PY'
from pathlib import Path
print('Container working directory:', Path.cwd())
Path('from_container.txt').write_text('Written inside the container\n')
PY
    cat from_container.txt
    grep -qx 'Written inside the container' from_container.txt
    echo PASS > bind_status.txt
    echo 'PASS: mounted code gave the same result and the host can read the output.'
else
    echo SKIPPED > bind_status.txt
    echo 'Bind example skipped: the checked baseline is not ready.'
fi
```

In [ ]:
run(r"""
source container.env
if [ "$(cat container_status.txt)" = READY ] && [ "$(cat repeat_status.txt)" = PASS ]; then
    echo FAILED > bind_status.txt
    srun -p cpu -N1 -n1 -c2 --mem=2G --time=00:03:00         apptainer exec --cleanenv --bind "$PWD:/work" --pwd /work         "$IMAGE" "$CONTAINER_PYTHON" train_model.py 42 3.0 | tee run_bind.json
    cmp run_A.json run_bind.json
    srun -p cpu -N1 -n1 -c1 --mem=1G --time=00:02:00         apptainer exec --cleanenv --bind "$PWD:/work" --pwd /work         "$IMAGE" "$CONTAINER_PYTHON" - <<'PY'
from pathlib import Path
print('Container working directory:', Path.cwd())
Path('from_container.txt').write_text('Written inside the container\n')
PY
    cat from_container.txt
    grep -qx 'Written inside the container' from_container.txt
    echo PASS > bind_status.txt
    echo 'PASS: mounted code gave the same result and the host can read the output.'
else
    echo SKIPPED > bind_status.txt
    echo 'Bind example skipped: the checked baseline is not ready.'
fi
""", check=False)

### Checkpoint 7

Identify the host path and container path in `--bind`. Explain why an output written under `/work` remains after the container process ends. `--cleanenv` controls environment inheritance; it is not a substitute for choosing where files are read and written.

## 10. Your next experiment

Change one item at a time and predict the effect before running it:

1. Use seed 7 with the same target slope. Does a new initialisation still converge to the intended line?
2. Increase the number of observations or training epochs in `train_model.py`. Keep the allocation fixed and record the change. Do not interpret this as GPU performance; the runs remain CPU-only.
3. Record the image's byte identity for a stricter replay. The command below reads the whole SIF file and can take time for a large image:

```bash
source container.env
sha256sum "$IMAGE" > image.sha256
cat run-identities.txt
```

A hash identifies file contents; it does not establish that an image came from a trusted source. A container does not freeze the host kernel, CPU, drivers or scheduler allocation.

Hand in the selected module list, compiler result, image/runtime information, A/B/C outputs, bind output and a short explanation of each comparison. The [extended notebook](M5S3_containers_portability_reproducibility_extended.ipynb) keeps the longer dependency, cache, numerical-library and handover exercises.

## Did it work?

The table below uses the status records written by the commands you ran. A skipped container stage is not a container success. The model table shows the values returned by actual executions; missing output is left missing.

In [ ]:
import json

stages = [
    ('Compiler stack selected', 'modules'),
    ('Native C result', 'native'),
    ('Module collection restored', 'restore'),
    ('HDF5 wrapper inspected', 'hdf5'),
    ('Container environment', 'container'),
    ('Repeat A versus B', 'repeat'),
    ('Changed input C', 'changed'),
    ('Bind and saved output', 'bind'),
]
print(f'{"Check":30s} Status')
for label, stage in stages:
    status = Path(stage + '_status.txt').read_text().strip()
    if stage == 'container' and status == 'READY': status = 'PASS'
    print(f'{label:30s} {status}')

print('\nRun   Target slope   Learned slope   Intercept   Mean absolute error')
for name in ['A', 'B', 'C']:
    path = Path('run_' + name + '.json')
    if not path.exists():
        print(name, '    No result produced')
        continue
    try:
        result = json.loads(path.read_text())
        print(f'{name:5s} {result["target_slope"]:12.3f} {result["learned_slope"]:15.6f} '
              f'{result["learned_intercept"]:11.6f} {result["mean_absolute_error"]:21.6f}')
    except (ValueError, KeyError):
        print(name, '    Output could not be read; inspect the command messages.')

References: [Lmod user guide](https://lmod.readthedocs.io/en/latest/010_user.html), [Apptainer execution](https://apptainer.org/docs/user/latest/quick_start.html), [PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html), and [bind mounts](https://apptainer.org/docs/user/latest/bind_paths_and_mounts.html).

The notebook does not install packages, build images or submit GPU jobs. Container execution is established only by the actual checks in your own run.